# 🤖 CoreTech RAG Knowledge Assistant
### Final Project — 100% Local, No API Key Required

---
✅ **Koi API key nahi chahiye**  
✅ **Koi signup nahi**  
✅ **Seedha Run All karo — sab kuch kaam karega**

---
**RAG Pipeline:**
```
User Query → TF-IDF Retriever → Top Docs → Rule-Based Answer Generator → Output
```

## Cell 1 — Libraries (Sirf Built-in, Koi Install Nahi)

In [ ]:
import re
import math
import json
from collections import Counter
from datetime import datetime

print('=' * 55)
print('  CoreTech RAG Knowledge Assistant')
print('  Final Project — No API Key Required')
print('=' * 55)
print('✅ All libraries loaded (100% built-in Python)')
print(f'   Started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Cell 2 — Knowledge Base Dataset (10 Documents)

In [ ]:
KNOWLEDGE_BASE = [
    {
        "id": "doc_001",
        "category": "Products",
        "title": "CoreTech CloudSuite Pro",
        "tags": ["cloud", "enterprise", "saas", "pricing", "storage", "users", "integration", "sla"],
        "content": (
            "CoreTech CloudSuite Pro is our flagship enterprise cloud platform for mid-to-large businesses. "
            "It provides a unified workspace for project management, team collaboration, data analytics, and CRM. "
            "CloudSuite Pro supports unlimited users on Enterprise tier and up to 50 users on Business tier. "
            "Pricing: $299/month for Business (50 users) and $899/month for Enterprise (unlimited users). "
            "Annual billing gives 20% discount. Integrates with Salesforce, HubSpot, Slack, Microsoft 365, Google Workspace. "
            "All tiers include 99.9% SLA uptime, SOC 2 Type II compliance, 24/7 technical support. "
            "Available on web, iOS, and Android. Data stored in US, EU, APAC ISO 27001 certified centers. "
            "Every subscription includes 1TB cloud storage. Extra storage is $0.10 per GB per month."
        )
    },
    {
        "id": "doc_002",
        "category": "Products",
        "title": "CoreTech DataBridge API",
        "tags": ["api", "integration", "developer", "data", "rest", "graphql", "sdk", "python", "rate limit"],
        "content": (
            "DataBridge API is CoreTech's developer-first integration platform for real-time data sync. "
            "Supports REST and GraphQL endpoints, webhooks, and event-driven architecture. "
            "Rate limits: 1,000 requests/minute on Standard, 10,000 on Premium. "
            "Standard plan: $199/month. Premium: $599/month. Enterprise custom plans available. "
            "SDK supports Python, Node.js, Java, Go, and .NET. "
            "Features: data transformation, schema mapping, automated retry logic. "
            "Latency SLA under 200ms (99th percentile, Premium). Sandbox environment free. "
            "All traffic encrypted with TLS 1.3. Supports OAuth 2.0 and API key auth."
        )
    },
    {
        "id": "doc_003",
        "category": "Products",
        "title": "CoreTech SecureVault",
        "tags": ["security", "encryption", "vault", "hipaa", "fips", "secrets", "compliance"],
        "content": (
            "SecureVault provides AES-256 encryption for credentials, API keys, and certificates. "
            "Integrates with Kubernetes, Docker, AWS, Azure, GCP via native plugins. "
            "Supports RBAC, audit logging, and automated secret rotation. "
            "Pricing: $0.05 per 10,000 API calls + $50/month base fee. "
            "FIPS 140-2 Level 3 certified. Supports HSMs. "
            "Free tier: 100,000 API calls/month. "
            "SOC 2 Type II, HIPAA, PCI-DSS compliance on all paid plans."
        )
    },
    {
        "id": "doc_004",
        "category": "Policies",
        "title": "Refund and Cancellation Policy",
        "tags": ["refund", "cancel", "billing", "policy", "money back", "termination", "guarantee"],
        "content": (
            "CoreTech offers a 30-day money-back guarantee on all new subscriptions. "
            "Cancel via customer portal or support@coretech.com. "
            "Monthly subscriptions: cancel any time, effective end of billing period. "
            "Annual subscriptions cancelled within 30 days: full refund. "
            "After 30 days: prorated refund minus 10% early termination fee. "
            "Refunds processed within 5-10 business days. "
            "Add-ons non-refundable once provisioned. "
            "Enterprise contracts governed by Master Service Agreement."
        )
    },
    {
        "id": "doc_005",
        "category": "Policies",
        "title": "Data Privacy and GDPR Compliance",
        "tags": ["privacy", "gdpr", "ccpa", "data", "compliance", "dpa", "export", "region"],
        "content": (
            "CoreTech is fully compliant with GDPR, CCPA, and PDPA. "
            "Signs Data Processing Agreement (DPA) with enterprise customers. "
            "Data stored in customer's selected region: US, EU, or APAC. "
            "Request data access, correction, deletion at privacy@coretech.com. "
            "Data retained 90 days after contract termination then permanently deleted. "
            "Export data in JSON or CSV from admin portal anytime. "
            "Annual third-party privacy audits. BYOK (Bring Your Own Key) on Enterprise."
        )
    },
    {
        "id": "doc_006",
        "category": "Support",
        "title": "Technical Support Plans",
        "tags": ["support", "sla", "helpdesk", "response time", "phone", "account manager", "ticket"],
        "content": (
            "Four support tiers available: "
            "Community: free, 5-7 business day email response. "
            "Standard: $99/month, 24-hour response, email and chat. "
            "Priority: $299/month, 4-hour response 24/7, phone support included. "
            "Dedicated: $999/month, named Technical Account Manager, 1-hour response 24/7, 2 on-site visits/year. "
            "P1 critical issues (full outage): response times halved on all paid plans. "
            "All paid plans include premium knowledge base and video tutorials."
        )
    },
    {
        "id": "doc_007",
        "category": "Support",
        "title": "Onboarding and Professional Services",
        "tags": ["onboarding", "setup", "implementation", "training", "certification", "migration"],
        "content": (
            "Standard onboarding free for all paid plans: 1-hour kickoff, videos, 30-day check-ins. "
            "Premium onboarding $1,500 one-time: 4-hour setup, data migration, 90-day plan. "
            "Enterprise onboarding $5,000+: dedicated engineer, workshops, 180-day plan. "
            "Certifications: Administrator ($299), Developer ($499), renewed annually. "
            "Group training (5+ employees): 20% discount."
        )
    },
    {
        "id": "doc_008",
        "category": "FAQ",
        "title": "Account Management FAQs",
        "tags": ["account", "login", "password", "mfa", "users", "sso", "saml", "audit", "reset"],
        "content": (
            "Password reset: visit login.coretech.com, click Forgot Password. Link sent in 2 minutes. "
            "MFA mandatory for admin accounts. Supports Google Authenticator, Authy, SMS, FIDO2 hardware keys. "
            "SSO via SAML 2.0 and OpenID Connect on Business and Enterprise plans. "
            "User seats added/removed anytime with prorated billing. "
            "Inactive users (90 days) auto-deactivated via Admin Console. "
            "Audit logs retained 12 months, exportable in CSV or SIEM format."
        )
    },
    {
        "id": "doc_009",
        "category": "FAQ",
        "title": "Integration and API FAQs",
        "tags": ["integration", "api", "webhook", "zapier", "slack", "salesforce", "github", "connect"],
        "content": (
            "200+ integrations: Salesforce, HubSpot, Slack, Teams, Jira, Asana, GitHub, Stripe, Xero. "
            "Zapier connects 5,000+ more apps without coding. "
            "Custom integrations via DataBridge API. "
            "Webhooks at Settings > Integrations > Webhooks as JSON over HTTPS. "
            "Failed webhooks retried 5 times with exponential backoff. "
            "Rate limit headers returned with every API response. IP whitelisting available."
        )
    },
    {
        "id": "doc_010",
        "category": "Pricing",
        "title": "Full Pricing and Plan Comparison",
        "tags": ["pricing", "plans", "compare", "cost", "billing", "startup", "nonprofit", "discount", "free"],
        "content": (
            "CloudSuite Starter: free, 5 users, 10GB storage, community support. "
            "CloudSuite Business: $299/month, 50 users, 1TB, standard support. "
            "CloudSuite Enterprise: $899/month, unlimited users, SSO, BYOK, custom SLAs. "
            "DataBridge Standard: $199/month. DataBridge Premium: $599/month. "
            "SecureVault Free: 100K calls/month. SecureVault paid: $50 base + $0.05/10K calls. "
            "Nonprofits and educational institutions: 30% discount. "
            "Startups under 2 years old and under $1M ARR: 50% off for 12 months via Startup Program. "
            "Volume discounts available above 500 users. All prices in USD."
        )
    }
]

print(f'✅ Knowledge Base Loaded — {len(KNOWLEDGE_BASE)} Documents')
print()
print(f'{"#":<6} {"ID":<10} {"Category":<12} {"Title"}')
print('-' * 60)
for i, doc in enumerate(KNOWLEDGE_BASE, 1):
    print(f'{i:<6} {doc["id"]:<10} {doc["category"]:<12} {doc["title"]}')

## Cell 3 — TF-IDF Retrieval Engine

In [ ]:
def tokenize(text):
    """Text ko lowercase tokens mein todta hai."""
    return [w for w in re.findall(r'\b[a-z]+\b', text.lower()) if len(w) > 2]


def compute_tfidf(query_tokens, doc):
    """
    TF-IDF score calculate karta hai document ke liye.
    Boosting:
      Title match  -> x5
      Tag match    -> x3
      Body match   -> x1 per occurrence
    """
    score = 0
    body   = Counter(tokenize(doc['content']))
    title  = Counter(tokenize(doc['title']))
    tags   = Counter(tokenize(' '.join(doc['tags'])))

    for token in query_tokens:
        score += body.get(token, 0)
        score += title.get(token, 0) * 5
        score += tags.get(token, 0)  * 3

    return score


def retrieve(query, top_k=3):
    """Query ke liye top-K relevant documents dhundta hai."""
    tokens = tokenize(query)
    scored = []
    for doc in KNOWLEDGE_BASE:
        s = compute_tfidf(tokens, doc)
        if s > 0:
            scored.append((s, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


def get_confidence(score):
    """Score ko confidence % mein convert karta hai."""
    if score >= 20: return 97
    if score >= 14: return 90
    if score >= 8:  return 80
    if score >= 4:  return 68
    return 50


print('✅ TF-IDF Retrieval Engine Ready')
print('   Title boost  : x5')
print('   Tag boost    : x3')
print('   Body         : x1 per occurrence')
print('   Top-K        : 3 documents')

## Cell 4 — Answer Generator (No API, Rule-Based + Extracted Text)

In [ ]:
def generate_answer(query, retrieved_docs):
    """
    Retrieved documents se direct answer extract karta hai.
    Koi API ki zaroorat nahi — pure Python.
    """
    if not retrieved_docs:
        return "Sorry, knowledge base mein is question ka jawab nahi mila."

    query_lower = query.lower()
    answer_parts = []

    for score, doc in retrieved_docs:
        sentences = re.split(r'(?<=[.!?])\s+', doc['content'])
        query_tokens = set(tokenize(query))

        # Har sentence ko score karo
        ranked = []
        for sent in sentences:
            sent_tokens = set(tokenize(sent))
            overlap = len(query_tokens & sent_tokens)
            if overlap > 0:
                ranked.append((overlap, sent.strip()))

        ranked.sort(reverse=True)
        top_sentences = [s for _, s in ranked[:3]]

        if top_sentences:
            answer_parts.append({
                'source': doc['title'],
                'category': doc['category'],
                'text': ' '.join(top_sentences)
            })

    return answer_parts


def display_confidence_bar(confidence):
    """Terminal mein confidence bar dikhata hai."""
    filled = int(confidence / 5)
    bar = '█' * filled + '░' * (20 - filled)
    return f'[{bar}] {confidence}%'


def rag_query(query):
    """
    Main RAG function:
    1. Retrieve
    2. Extract relevant sentences
    3. Display with sources and confidence
    """
    print()
    print('=' * 60)
    print(f'🔍 QUERY: {query}')
    print('=' * 60)

    # Step 1: Retrieve
    retrieved = retrieve(query, top_k=3)

    if not retrieved:
        print('⚠️  Koi relevant document nahi mila.')
        return

    top_score = retrieved[0][0]
    confidence = get_confidence(top_score)

    print(f'\n📄 Retrieved Documents ({len(retrieved)}):')
    for rank, (score, doc) in enumerate(retrieved, 1):
        print(f'   {rank}. {doc["title"]} [{doc["category"]}] — Score: {score}')

    print(f'\n📊 Confidence: {display_confidence_bar(confidence)}')

    # Step 2: Generate answer
    answer_parts = generate_answer(query, retrieved)

    print(f'\n🤖 ANSWER:')
    print('-' * 60)

    if isinstance(answer_parts, str):
        print(answer_parts)
    else:
        for part in answer_parts:
            print(f'\n📌 From: {part["source"]} [{part["category"]}]')
            print(f'   {part["text"]}')

    print('\n' + '-' * 60)
    print(f'✅ Sources: {" | ".join([doc["title"] for _, doc in retrieved])}')
    print('=' * 60)


print('✅ Answer Generator Ready (No API Required)')

## Cell 5 — Demo: 6 Sample Queries

In [ ]:
print('🚀 DEMO — 6 SAMPLE QUERIES')

sample_queries = [
    "What are the pricing plans for CloudSuite Pro?",
    "Is CoreTech GDPR compliant and how is data protected?",
    "What is the refund policy for annual subscriptions?",
    "What support plans are available and what are response times?",
    "What programming languages does DataBridge SDK support?",
    "Are there discounts for nonprofits or startups?"
]

for q in sample_queries:
    rag_query(q)

## Cell 6 — Apna Khud Ka Sawaal Pocho

In [ ]:
# ✏️  Yahan apna sawaal likhein aur cell run karein
my_question = "How do I reset my password and set up MFA?"

rag_query(my_question)

## Cell 7 — Batch Evaluation (Saare Queries Ek Saath)

In [ ]:
TEST_QUERIES = [
    "CloudSuite Pro pricing",
    "GDPR compliance data privacy",
    "refund policy cancel subscription",
    "technical support plans response time",
    "DataBridge API python sdk developer",
    "nonprofit startup discount pricing",
    "MFA password reset account login",
    "webhook integration zapier salesforce",
    "SecureVault encryption HIPAA security",
    "onboarding training certification"
]

print('🔬 BATCH EVALUATION RESULTS')
print('=' * 70)
print(f'{"#":<4} {"Query":<40} {"Top Doc":<20} {"Conf"}')
print('-' * 70)

total_conf = 0
results = []

for i, q in enumerate(TEST_QUERIES, 1):
    retrieved = retrieve(q, top_k=3)
    if retrieved:
        score, doc = retrieved[0]
        conf = get_confidence(score)
        top_doc = doc['title'][:20]
    else:
        conf = 0
        top_doc = 'None'
    total_conf += conf
    results.append({'query': q, 'confidence': conf, 'docs': len(retrieved)})
    print(f'{i:<4} {q[:40]:<40} {top_doc:<20} {conf}%')

avg = total_conf / len(TEST_QUERIES)
print('=' * 70)
print(f'\n📊 SUMMARY:')
print(f'   Total Queries     : {len(TEST_QUERIES)}')
print(f'   Average Confidence: {avg:.1f}%')
print(f'   All Answered      : {sum(1 for r in results if r["docs"] > 0)}/{len(TEST_QUERIES)}')
print(f'   Retrieval Method  : TF-IDF with Title+Tag boosting')

## Cell 8 — RAG vs Direct Comparison (Bina API)

In [ ]:
def direct_answer_simple(query):
    """Bina retrieval ke generic answer — hallucination ka risk hota hai."""
    return (
        f"(Direct Mode — No retrieval used)\n"
        f"I am the CoreTech assistant. Regarding '{query}', "
        f"I would need to check the documentation. "
        f"Without retrieval, specific prices and policies may not be accurate."
    )


compare_query = "What is the price of CloudSuite Enterprise and what does it include?"

print('🆚 RAG MODE vs DIRECT MODE')
print('=' * 60)
print(f'Query: {compare_query}')
print()

# RAG Mode
print('✅ RAG MODE (Knowledge Base Se Grounded Answer):')
print('-' * 60)
retrieved = retrieve(compare_query, top_k=2)
answer_parts = generate_answer(compare_query, retrieved)
for part in answer_parts:
    print(f'Source: {part["source"]}')
    print(f'{part["text"]}')
    print()

# Direct Mode
print('⚠️  DIRECT MODE (No Retrieval — Hallucination Risk):')
print('-' * 60)
print(direct_answer_simple(compare_query))
print()
print('=' * 60)
print('📌 Conclusion: RAG mode gives specific, accurate, cited answers.')
print('   Direct mode risks making up incorrect information.')

## Cell 9 — Save Report as JSON File

In [ ]:
report = {
    "project": "CoreTech RAG Knowledge Assistant",
    "type": "Final Project",
    "mode": "No API — 100% Local Python",
    "generated_at": datetime.now().isoformat(),
    "knowledge_base": {
        "total_documents": len(KNOWLEDGE_BASE),
        "categories": list(set(d['category'] for d in KNOWLEDGE_BASE)),
        "documents": [{"id": d["id"], "title": d["title"], "category": d["category"]} for d in KNOWLEDGE_BASE]
    },
    "retrieval_engine": {
        "method": "TF-IDF",
        "title_boost": 5,
        "tag_boost": 3,
        "top_k": 3
    },
    "evaluation": {
        "total_queries": len(results),
        "average_confidence": round(avg, 1),
        "all_answered": sum(1 for r in results if r['docs'] > 0),
        "details": results
    }
}

with open('coretech_rag_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('✅ Report saved: coretech_rag_report.json')
print(f'   Documents : {report["knowledge_base"]["total_documents"]}')
print(f'   Avg Conf  : {report["evaluation"]["average_confidence"]}%')
print(f'   Answered  : {report["evaluation"]["all_answered"]}/{report["evaluation"]["total_queries"]}')
print()
print('📥 Download: Files panel (left) → right-click coretech_rag_report.json → Download')

---
## ✅ Project Complete!

| Component | Detail |
|---|---|
| **Dataset** | 10 documents, 5 categories (built-in) |
| **Retrieval** | TF-IDF with title (x5) and tag (x3) boosting |
| **Generation** | Sentence extraction from retrieved docs |
| **API Key** | ❌ Not required |
| **Install** | ❌ Nothing to install |
| **Modes** | RAG vs Direct comparison |
| **Output** | Answer + Sources + Confidence + JSON report |

---
*CoreTech RAG Knowledge Assistant — Final Project (No API Version)*